# Generate Embeddings

# Imports

In [32]:
from itertools import combinations
import networkx as nx
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
from scipy.stats import spearmanr
from tqdm import tqdm as tqdm

import torch
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
from torch_geometric.utils import from_networkx
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU
from sklearn.metrics.pairwise import cosine_similarity


# Directories

In [18]:
# Define current directory
cwd = Path.cwd()
# Define DATA directory
DATA = cwd.parents[1]/'data'/'canada'

# Define INPUT directory
INPUT = DATA / 'input'

# Define OUTPUT directory
OUTPUT = DATA / 'output'

# Define CONTEXT directory
CONTEXT = OUTPUT / 'context_graphs'

# Define RNASEQ directory
RNASEQ = CONTEXT / 'rnaseq'

# Define CID directory
CID = CONTEXT / 'cid'

# Define MEAN directory
MEAN = CONTEXT / 'mean'

# Functions

## General

In [6]:
def file_to_list(path):
    '''
    Converts a .txt file to a list
    '''

    with open(f'{path}', 'r', encoding = 'utf-8') as f:
        list_file = [line.strip() for line in f]
    
    return list_file

def list_to_file(path, data):
      '''
      Saves a list or set to a .txt file with no header.
      '''

      with open(path, 'w') as f:
            for item in sorted(data):
                  f.write(f'{item}\n')

def pickle_load(path: str, report: bool = False):
    '''
    Loads pickled data.
    '''

    with open(path, 'rb') as f:
        data = pickle.load(f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_load')
            print(f'Pickled graph loaded w/ {num_nodes:,} nodes and {num_edges:,} edges')
            print()
        else:
            print('>> pickle_load')
            print(f'Pickled file loaded')
            print()

    return data

def pickle_save(path: str, data, report: bool = False):
    '''
    Pickles data.
    '''

    with open(path, 'wb') as f:
        pickle.dump(data, f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_save')
            print(f'Graph w/ {num_nodes:,} nodes and {num_edges:,} edges pickled')
        else:
            print('>> pickle_save')
            print(f'Data pickled')
            print()

## GIN

In [34]:
def gin_jaccard_similarity(set1, set2):
    return len(set1 & set2) / len(set1 | set2)

def gin_upper_tri(mat):
    idx = np.triu_indices_from(mat, k=1)
    
    return mat[idx]

def preprocess_graphs(list_pyg):
    '''
    Applies only common attributes to PyG graph objects (some graphs have extra 'dose' attribute that required removal)
    '''
    keys_to_keep = ['x', 'edge_index', 'num_nodes', 'name', 'timepoint']
    for g in list_pyg:
        # Remove unnecessary attributes
        for key in list(g.keys()):  # <-- call keys() as a method
            if key not in keys_to_keep:
                del g[key]

        # Make x 2D
        if g.x.dim() == 1:
            g.x = g.x.view(-1, 1)
    return list_pyg

class GINEncoder(torch.nn.Module):
    '''
    GIN encoder with message passing, a convolutional layer and a global pooling layer to generate a graph embedding
    '''
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=3):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        for i in range(num_layers):
            in_dim = input_dim if i == 0 else hidden_dim
            mlp = Sequential(Linear(in_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
            conv = GINConv(mlp)
            self.convs.append(conv)

    def forward(self, x, edge_index, batch):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
        graph_emb = global_add_pool(x, batch)  # graph-level embedding
        return graph_emb

def info_nce_loss(z1, z2, temperature=0.5):
    '''
    Information Noise-Contrastive Estimation as a loss function
    '''
    # Normalize embeddings
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    
    batch_size = z1.size(0)
    
    # Cosine similarity matrix
    representations = torch.cat([z1, z2], dim=0)  # [2*B, D]
    sim_matrix = torch.matmul(representations, representations.T)  # [2B,2B]
    
    # Scale by temperature
    sim_matrix = sim_matrix / temperature
    
    # Labels: each sample i in z1 matches i+B in z2, and vice versa
    labels = torch.arange(batch_size, device=z1.device)
    
    # Mask out self-similarities
    mask = torch.eye(2*batch_size, device=z1.device).bool()
    sim_matrix.masked_fill_(mask, -9e15)  # very large negative number instead of -inf
    
    # Positive logits
    positives = torch.cat([torch.arange(batch_size, batch_size*2), torch.arange(0, batch_size)]).to(z1.device)
    
    # Cross-entropy
    loss = F.cross_entropy(sim_matrix, positives)
    return loss

def augment_graph_batch(batch_graphs, noise_scale=0.1):
    '''
    Graph batch augmentation
    '''
    x_aug = batch_graphs.x + noise_scale * torch.randn_like(batch_graphs.x)
    batch_aug = Batch(batch_graphs.to_data_list())
    batch_aug.x = x_aug.to(batch_graphs.x.device)
    batch_aug.edge_index = batch_graphs.edge_index
    batch_aug.batch = batch_graphs.batch
    return batch_aug

def train_contrastive_gin(list_pyg, embed_dim=64, epochs=30, batch_size=4, lr=1e-3, device=None):
    '''
    Implementaion of GIN w/ contrastive learning loss function
    '''
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

    # list_pyg = strip_graph_attributes(list_pyg)
    loader = DataLoader(list_pyg, batch_size=batch_size, shuffle=True)
    encoder = GINEncoder(input_dim=1, hidden_dim=embed_dim).to(device)
    optimizer = torch.optim.Adam(encoder.parameters(), lr=lr)

    #for epoch in tqdm(range(epochs), desc = f'Training contrastive GIN', total = epochs):
    for epoch in range(epochs):
        encoder.train()
        total_loss = 0
        #for batch_graphs in tqdm(loader, desc=f'Epoch {epoch+1}/{epochs}'):
        for batch_graphs in loader:
            batch_graphs = batch_graphs.to(device)
            batch_aug = augment_graph_batch(batch_graphs)

            z1 = encoder(batch_graphs.x, batch_graphs.edge_index, batch_graphs.batch)
            z2 = encoder(batch_aug.x, batch_aug.edge_index, batch_aug.batch)

            loss = info_nce_loss(z1, z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        
        # Report loss per epoch
        #print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

    return encoder

def get_graph_embeddings(encoder, list_pyg, device=None):
    '''
    Generates graph embeddings using trained contrastive GIN
    '''
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    encoder.eval()
    embeddings = []
    graph_info = []
    with torch.no_grad():
        #for g in tqdm(list_pyg, desc="Generating embeddings"):
        for g in list_pyg:
            g = g.to(device)
            batch = torch.zeros(g.num_nodes, dtype=torch.long, device=device)
            z = encoder(g.x, g.edge_index, batch)
            embeddings.append(z.cpu())
            graph_info.append({'name': g.name, 'timepoint': g.timepoint})
    embeddings = torch.cat(embeddings, dim=0).numpy()
    return embeddings, graph_info

# Graph Embedding

Done using GIN + contrastive learning

## PyG Graph List

To pass relevant graphs to PyTorch, they are converted to a list of PyG graph objects.

In [20]:
# Initialise list
list_pyg = []

### RNAseq Graphs

In [21]:
# Iterate through RNAseq graphs
for file in tqdm(RNASEQ.iterdir(), desc = 'Converting RNAseq graphs to PyG', total = 3):
    # Get filename
    filename = str(file).split('\\')[-1]
    desc = filename.split('.pkl')[0]

    # Filter for 6H data (TO BE REMOVED LATER)
    if '6H' in filename:
        
        # Get metadata
        timepoint = desc.split('_')[0]
        perturbagen_id = desc.split('_')[1]
        dose = '10'

        # Load graph
        graph = pickle_load(file)

        # Convert to PyG object
        pyg = from_networkx(graph)
        # Format expression attribute
        pyg.x = pyg.x.float()
        # Add perturbagen id
        pyg.name = f'{perturbagen_id}'
        # Add dose
        pyg.dose = f'{dose}'
        # Add timepoint
        pyg.timepoint = f'{timepoint}'

        # Append to graph list
        list_pyg.append(pyg)

Converting RNAseq graphs to PyG: 6it [00:00,  7.13it/s]               


### Context Graphs

In [22]:
# Iterate through context graphs
for file in tqdm(MEAN.iterdir(), desc = 'Converting context graphs to Pyg', total = 100):
    
    # Get metadata
    desc = str(file).split('\\')[-1]
    desc = desc.split('.pkl')[0]
    perturbagen_id = desc.split('_')[0]
    cell_line = desc.split('_')[1]
    dose = desc.split('_')[2]
    timepoint = desc.split('_')[3]

    # Load graph
    graph = pickle_load(file)

    # Convert to PyG object
    pyg = from_networkx(graph)
    # Format expression attribute
    pyg.x = pyg.x.float()
    # Add perturbagen id
    pyg.name = f'{perturbagen_id}'
    # Add dose
    pyg.dose = f'{dose}'
    # Add timepoint
    pyg.timepoint = f'{timepoint}'

    # Append to graph list
    list_pyg.append(pyg)

Converting context graphs to Pyg: 100%|██████████| 100/100 [00:29<00:00,  3.38it/s]


## Run GIN

### Preprocess Graphs

Checks shape of expression attribute and reshapes if necessary.

In [42]:
# Preprocess list_pyg
list_pyg = preprocess_graphs(list_pyg)
# Save list_pyg
pickle_save(OUTPUT / 'list_pyg.pkl', list_pyg)
print('Preprocessing complete')

Preprocessing complete


### Multiple Runs

In [35]:
# Define number of runs
NUM_RUNS = 20

In [41]:
# Initialise embeddings
list_emb = []
# Initialise similarity matrices
sim_matrices = []
# Initialise graph info
list_graph_info = []

# Multiple runs
for seed in tqdm(range(NUM_RUNS), desc = 'Running GIN', total = NUM_RUNS):

    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Define encoder
    encoder = train_contrastive_gin(list_pyg, embed_dim=64, epochs=150, batch_size=32)
    # Generate embeddings and graph info
    embeddings, graph_info = get_graph_embeddings(encoder, list_pyg)

    # Center and normalize
    emb_centered = embeddings - embeddings.mean(axis=0, keepdims=True)
    emb_norm = emb_centered / np.linalg.norm(emb_centered, axis=1, keepdims=True)
    # Append embeddings
    list_emb.append(emb_norm)

    # Calculate cosine similarity between normalised embeddings
    sim_matrices.append(cosine_similarity(emb_norm))

    # Append graph info
    list_graph_info = graph_info

# Save list_emb for analysis
pickle_save(OUTPUT / 'list_emb.pkl', list_emb)

Running GIN: 100%|██████████| 20/20 [03:42<00:00, 11.13s/it]


### Run Similarity

Given a reference treatment, how similar are multiple runs based on similarity metrics:
- Pearson Correlation
- Spearman Correlation
- Jaccard Index

In [37]:
# Define reference treatment
REF_TREATMENT = 'Nita'

# Get reference treatment graph index
ref_idx = next(index for index, graph in enumerate(list_graph_info) if REF_TREATMENT in graph['name'])

# Generate empty Pearson correlation matrix
corr_matrix = np.zeros((NUM_RUNS, NUM_RUNS))
# Generate empty Spearman correlation matrix
spearman_matrix = np.zeros((NUM_RUNS, NUM_RUNS))
# Generate empty Jaccard matrix
jaccard_matrix = np.zeros((NUM_RUNS, NUM_RUNS))

# Iterate through runs
for i in range(NUM_RUNS):
    for j in range(i + 1, NUM_RUNS):
        
        # Pearson correlation
        corr = np.corrcoef(gin_upper_tri(sim_matrices[i]), gin_upper_tri(sim_matrices[j]))[0,1]
        corr_matrix[i, j] = corr_matrix[j, i] = corr

        # Spearman rank correlation
        rho, _ = spearmanr(sim_matrices[i][ref_idx], sim_matrices[j][ref_idx])
        spearman_matrix[i, j] = spearman_matrix[j, i] = rho

# Compute mean pearson value
mean_pearson = corr_matrix.mean(axis = 1)
# Compute mean spearman value
mean_spearman = spearman_matrix.mean(axis = 1)

# Define number of top graphs to compare
k = 5
# Iterate through combinations of runs
for i,j in combinations(range(NUM_RUNS), 2):
    
    # Get top k closest neighbours to reference treatment
    top_i = set(np.argsort(-sim_matrices[i][ref_idx])[:k])
    top_j = set(np.argsort(-sim_matrices[j][ref_idx])[:k])
    
    # Calculate jaccard similarity
    jsimilarity = gin_jaccard_similarity(top_i, top_j)
    # Convert to matrix
    jaccard_matrix[i, j] = jaccard_matrix[j, i] = jsimilarity

# Calculate mean jsimilarity per run
mean_jaccard = jaccard_matrix.mean(axis = 1)

# Generate dataframe
df_consistency = pd.DataFrame({'run' : range(NUM_RUNS),
                               'mean_pearson' : mean_pearson,
                               'mean_spearman' : mean_spearman,
                               'mean_jaccard' : mean_jaccard})

# Calculate combined score
df_consistency['combined_score'] = (0.5 * (df_consistency['mean_pearson'] + df_consistency['mean_spearman']) +
                                    0.5 * df_consistency['mean_jaccard'])

# Sort values
df_consistency.sort_values(by = 'combined_score', ascending = False, inplace = True, ignore_index = True)
# Show data
df_consistency.head(10)

,run,mean_pearson,mean_spearman,mean_jaccard,combined_score
0,15,0.616709,0.329790,0.496032,0.721265
1,4,0.610481,0.353514,0.476587,0.720291
2,0,0.586446,0.326358,0.489484,0.701144
3,17,0.617138,0.224753,0.496032,0.668961
4,1,0.469593,0.365766,0.496032,0.665696
5,2,0.604121,0.367737,0.358532,0.665195
6,13,0.545795,0.282241,0.485317,0.656677
7,16,0.542698,0.369247,0.349603,0.630774
8,5,0.498246,0.331753,0.426984,0.628492
9,10,0.563209,0.333553,0.342659,0.619711


### Consensus Embedding

The top 5 most similar runs are selected, and their embeddings averaged to produce the final embedding.

In [38]:
# Load perturbagen data
df_perturbagen_info = pd.read_csv(OUTPUT / 'df_perturbagen_info.csv')
# Show data
df_perturbagen_info.head()

,perturbagen_id,perturbagen_name
0,DMSO,DMSO
1,BRD-A03772856,BRD-A03772856
2,BRD-A19037878,trichostatin-a
3,BRD-A19500257,geldanamycin
4,BRD-A34037822,KUC107191N


In [40]:
# Select top 5 runs based on combined_score
list_top_runs = df_consistency['run'][0:5].values

# Extract embeddings generated by those runs
list_cosine = [sim_matrices[run] for run in list_top_runs]

# Stack extracted cosine matrics
cosine_stacked = np.stack(list_cosine, axis = 0)
# Compute consensus cosine similarities (mean across runs)
consensus = np.mean(cosine_stacked, axis = 0)

# Define n closest neighbours
N_CLOSEST = 10
# Define REF_TREATMENT
REF_TREATMENT = 'Nita'
# Get REF_TREATMENT index
ref_idx = next(index for index, graph in enumerate(list_graph_info) if REF_TREATMENT in graph['name'])

# Get index of closest neighbours
#top_idx = np.argsort(consensus[ref_idx])[::-1][1:(N_CLOSEST + 1)]
top_idx = np.argsort(consensus[ref_idx])[::-1][1:]

# Get data
list_data = [(list_graph_info[i]['name'], list_graph_info[i]['timepoint'], consensus[ref_idx, i]) for i in top_idx]

# Generate dataframe
df_consensus = pd.DataFrame(list_data, columns = ['name', 'timepoint', 'cosim'])

# Rename columns
df_consensus.rename(columns = {'name': 'perturbagen_id'}, inplace = True)
# Merge w/ df_perturbagen
df_consensus = pd.merge(df_consensus, df_perturbagen_info, how = 'left', on = 'perturbagen_id')
# Replace NaN
df_consensus['perturbagen_name'] = df_consensus['perturbagen_name'].fillna(df_consensus['perturbagen_id'])

# Show data
df_consensus.head(10)

,perturbagen_id,timepoint,cosim,perturbagen_name
0,Paro,6H,0.992053,Paro
1,BRD-K88172511,6H,0.987887,naltrexone
2,BRD-K17743125,6H,0.914016,belinostat
3,BRD-K68202742,6H,0.856158,trichostatin-a
4,BRD-K02130563,6H,0.835377,panobinostat
5,BRD-A19037878,6H,0.802536,trichostatin-a
6,BRD-A44090213,6H,0.798061,indoprofen
7,BRD-K81418486,6H,0.779195,vorinostat
8,BRD-K22503835,6H,0.700619,scriptaid
9,BRD-K66404838,6H,0.512967,nalbuphine
